# v13: Seed-averaging the models (variance reduction)

**Version:** v13  
**Kaggle submission:** `v13_bag` → `outputs/submission_v13_bag.csv` (single change vs `v12_robust`)

**Question:** v12_robust beat v10_bayes by 509 public points. Is that a real gain from the cost-model inputs, or a lucky draw of XGBoost's random column/row sampling? A single fit's test predictions change as much between seeds as between the two versions we compared. We therefore average the classifier over 5 seeds and the repair-cost model over 8 seeds and change nothing else.

*Self-contained: needs only `../data/train.csv` and `../data/test.csv`. Part of the Fleet Maintenance project (Kaggle `mlprojectfleet2`); the cost, decision rule and validation protocol are derived in `fleet_maintenance.ipynb`.*

In [1]:
import warnings; warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, xgboost as xgb
from pathlib import Path
from scipy.stats import ks_2samp
from sklearn.isotonic import IsotonicRegression
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, average_precision_score, r2_score, brier_score_loss, log_loss
from sklearn.model_selection import StratifiedKFold, KFold, train_test_split, cross_val_predict

DATA, OUT = Path("../data"), Path("outputs"); OUT.mkdir(exist_ok=True)
SEED = 42
INSPECT, UNDER, OVER = 10.0, 0.20, 0.02          # inspection cost; shortage cost / unit; unused-budget cost / unit
Q_STAR = UNDER / (UNDER + OVER)                  # newsvendor critical ratio 0.909
HISTB = [f"s{i:03d}" for i in range(54, 64)]     # 10-bin histogram that is shifted by one bin in test

def load(raw=False):
    """train/test with the v1 fixes: realign s054-s063 (test bin k+1 == train bin k), drop s047 (all NaN in
    test) and s146 (constant). raw=True returns the files untouched."""
    tr, te = pd.read_csv(DATA / "train.csv"), pd.read_csv(DATA / "test.csv")
    if not raw:
        te[HISTB[:-1]] = te[HISTB[1:]].to_numpy(); te[HISTB[-1]] = np.nan
        tr, te = tr.drop(columns=["s047", "s146"]), te.drop(columns=["s047", "s146"])
    return tr, te, [c for c in tr.columns if c.startswith("s")]

tr, te, S = load()
dev_idx, ho_idx = train_test_split(np.arange(len(tr)), test_size=0.2, stratify=tr.failure, random_state=SEED)
dev, ho = tr.iloc[dev_idx].reset_index(drop=True), tr.iloc[ho_idx].reset_index(drop=True)   # ho: locked 20% hold-out
y, c = dev.failure.to_numpy(), dev.repair_cost.fillna(0).to_numpy() * dev.failure.to_numpy()
y_ho, c_ho = ho.failure.to_numpy(), ho.repair_cost.fillna(0).to_numpy() * ho.failure.to_numpy()
c_all = tr.repair_cost.fillna(0).to_numpy() * tr.failure.to_numpy()
F = list(StratifiedKFold(5, shuffle=True, random_state=SEED).split(np.zeros(len(y)), y))   # the 5 fixed folds
print(f"train {tr.shape}  test {te.shape}  sensors {len(S)}  dev {len(dev)} / hold-out {len(ho)}  failures {int(y.sum())}")

train (56000, 172)  test (20000, 169)  sensors 168  dev 44800 / hold-out 11200  failures 318


In [2]:
def vehicle_cost(c, flag, b):
    return np.where(flag == 1, INSPECT + UNDER * np.maximum(c - b, 0) + OVER * np.maximum(b - c, 0), c)

def per_vehicle(c, flag, b):
    return float(vehicle_cost(np.asarray(c, float), np.asarray(flag), np.asarray(b, float)).mean())

fold_cost = lambda flag, b: np.array([per_vehicle(c[v], flag[v], b[v]) for _, v in F])

def write_submission(ids, flag, budget, name):
    flag, budget = np.asarray(flag).astype(int), np.maximum(np.asarray(budget, float), 0)
    assert len(ids) == 20000 and ids.is_unique and set(flag) <= {0, 1} and np.isfinite(budget).all()
    pd.DataFrame({"id": ids, "flag": flag, "budget": budget.round(2)}).to_csv(OUT / f"submission_{name}.csv", index=False)
    print(f"wrote outputs/submission_{name}.csv: flagged {flag.mean():.2%} of test vehicles")

def decide_global(p, fail_costs):
    """Same repair-cost distribution F (empirical failure costs) for every vehicle."""
    p = np.clip(np.asarray(p, float), 0, 1); fc = np.sort(np.asarray(fail_costs, float)); mu = fc.mean()
    b = np.zeros_like(p); need = p > 1 - Q_STAR
    b[need] = np.quantile(fc, 1 - (1 - Q_STAR) / p[need])                       # newsvendor budget
    fail = (UNDER * np.maximum(fc[None, :] - b[:, None], 0) + OVER * np.maximum(b[:, None] - fc[None, :], 0)).mean(1)
    return ((INSPECT + p * fail + (1 - p) * OVER * b) < p * mu).astype(int), b   # flag iff cheaper than not flagging

def decide_pv(p, m, resid, fail_costs=None, K=99, scale=None):
    """Per-vehicle cost distribution: log c | fail ~ m_i + scale_i * residuals (out-of-fold residuals).
    fail_costs given -> budget from that GLOBAL distribution (ablation); scale -> per-vehicle spread (v9)."""
    p, m = np.clip(np.asarray(p, float), 0, 1), np.asarray(m, float)
    sc = np.ones_like(m) if scale is None else np.asarray(scale, float)
    M = np.exp(m[:, None] + sc[:, None] * np.quantile(resid, (np.arange(K) + 0.5) / K)[None, :]); mu = M.mean(1)
    b = np.zeros_like(p); need = p > 1 - Q_STAR; level = 1 - (1 - Q_STAR) / p[need]
    b[need] = (np.exp(m[need] + sc[need] * np.quantile(resid, level)) if fail_costs is None
               else np.quantile(np.asarray(fail_costs, float), level))
    fail = (UNDER * np.maximum(M - b[:, None], 0) + OVER * np.maximum(b[:, None] - M, 0)).mean(1)
    return ((INSPECT + p * fail + (1 - p) * OVER * b) < p * mu).astype(int), b

class Platt:
    def fit(self, p, y):
        self.m = LogisticRegression(C=1e6, max_iter=1000).fit(self._x(p), y); return self
    def predict(self, p): return self.m.predict_proba(self._x(p))[:, 1]
    @staticmethod
    def _x(p): p = np.clip(p, 1e-6, 1 - 1e-6); return np.log(p / (1 - p))[:, None]

class Iso:
    def fit(self, p, y):
        self.m = IsotonicRegression(out_of_bounds="clip", y_min=0, y_max=1).fit(p, y); return self
    def predict(self, p): return self.m.predict(p)

make_cal = lambda kind: {"platt": Platt, "iso": Iso}[kind]()

PARAMS = dict(n_estimators=300, max_depth=4, learning_rate=0.05, subsample=0.8, colsample_bytree=0.8,
              min_child_weight=1, n_jobs=-1, random_state=SEED)                         # failure classifier (v2..)
CM = dict(n_estimators=200, max_depth=3, learning_rate=0.03, subsample=0.8, colsample_bytree=0.5,
          min_child_weight=5, n_jobs=-1, random_state=SEED)                             # repair-cost model (v5)
inner = KFold(5, shuffle=True, random_state=SEED)

def cost_model(params, Xa, lc, Xb):
    """XGBoost on log repair cost (failures only). Returns predictions for Xb and OUT-OF-FOLD residuals of Xa."""
    resid = lc - cross_val_predict(xgb.XGBRegressor(**params), Xa, lc, cv=inner)
    return xgb.XGBRegressor(**params).fit(Xa, lc).predict(Xb), resid

def fit_oof(X, params=PARAMS):
    """5-fold out-of-fold failure probabilities on dev, and probabilities for test from a model fit on all train."""
    o = np.zeros(len(dev))
    for a, b in F:
        o[b] = xgb.XGBClassifier(**params).fit(X.iloc[a], y[a]).predict_proba(X.iloc[b])[:, 1]
    return o

shift = lambda p, k: k * p / (1 - p) / (1 + k * p / (1 - p))       # prior shift by odds factor k

In [3]:
HIST = {a: [f"s{i:03d}" for i in range(a, a + 10)] for a in (23, 37, 54, 80, 129, 154)}   # six hidden 10-bin histograms
HCOLS = {col for g in HIST.values() for col in g}

def shape(df):
    """Histogram-shape features: per-bin shares, mean/std bin, entropy, low/high-3 tail shares, log usage."""
    out = {"logU": np.log1p(df[HIST[54][:9]].sum(axis=1, min_count=9))}
    for a, g in HIST.items():
        g = g[:9] if a == 54 else g                        # test s063 is NaN after realignment
        x = df[g].to_numpy(float); tot = x.sum(1, keepdims=True)
        sh = np.where(tot > 0, x / np.where(tot > 0, tot, 1), np.nan); k = np.arange(len(g)); mean = (sh * k).sum(1)
        out.update({f"h{a}_sh{i}": sh[:, i] for i in range(len(g))})
        out[f"h{a}_mean"] = np.where(tot[:, 0] > 0, mean, np.nan)
        out[f"h{a}_std"] = np.where(tot[:, 0] > 0, np.sqrt(np.maximum((sh * k ** 2).sum(1) - mean ** 2, 0)), np.nan)
        with np.errstate(divide="ignore", invalid="ignore"):
            out[f"h{a}_ent"] = np.where(tot[:, 0] > 0, -np.nansum(np.where(sh > 0, sh * np.log(sh), 0), 1), np.nan)
        out[f"h{a}_lo3"], out[f"h{a}_hi3"] = sh[:, :3].sum(1), sh[:, -3:].sum(1)
    return pd.DataFrame(out, index=df.index)

def ratios(df, cols):
    U = df[HIST[54][:9]].sum(axis=1, min_count=9).replace(0, np.nan)
    return df[[col for col in cols if col not in HCOLS]].div(U, axis=0).add_suffix("_perU")

In [4]:
CM9 = {**CM, "n_estimators": 1000, "max_depth": 2}                      # v9 cost model
DRIFT = ["s120", "s122", "s126", "s128"]; MAP = ["s120", "s122"]
keep = [s for s in S if s not in DRIFT]
f_all = tr[tr.failure == 1]; lc_all = np.log(f_all.repair_cost.to_numpy())
Xf, Xt = pd.concat([f_all[keep], shape(f_all)], axis=1), pd.concat([te[keep], shape(te)], axis=1)   # cost model: no drifted sensors (v12)
# posterior over the depot shift k from the v10 notebook (nine public scores), full precision
GRID = [1.5, 2.0, 2.5, 3.0, 3.5, 4.0, 4.5, 5.0, 5.5, 6.0, 7.0, 8.0, 10.0, 12.0]
W = [0.008172761879954695, 0.13116481847950048, 0.26721015776942314, 0.267056932957599, 0.1710924552535038, 0.0956936236265262,
     0.03459275261241694, 0.016549303552639335, 0.005725443007721319, 0.002368204370941116, 0.00031775367779660887,
     5.437858042612226e-05, 1.3770078232241008e-06, 3.722372808678639e-08]
pavg = lambda qq, lam=1.0: sum(w * shift(qq, k * lam) for w, k in zip(W, GRID))   # posterior-averaged shifted probability

### 1. How large is seed-to-seed noise compared with the effect we were trying to measure?

In [5]:
NK = 8
m_seeds, mo = [], np.zeros(len(f_all))                                   # cost model: NK seeds, bagged inner out-of-fold residuals
for s in range(NK):
    prm = {**CM9, "random_state": SEED + s}
    mo += cross_val_predict(xgb.XGBRegressor(**prm), Xf, lc_all, cv=inner) / NK
    m_seeds.append(xgb.XGBRegressor(**prm).fit(Xf, lc_all).predict(Xt))
m_bag, resid_bag = np.mean(m_seeds, axis=0), lc_all - mo

dm = lambda a, b: float(np.abs(a - b).mean())
pairs = [dm(m_seeds[i], m_seeds[j]) for i in range(NK) for j in range(i + 1, NK)]
m_drift_all, _ = cost_model(CM9, pd.concat([f_all[S], shape(f_all)], axis=1), lc_all, pd.concat([te[S], shape(te)], axis=1))
print(f"mean abs difference of test log-cost predictions between two single fits (different seeds): {np.mean(pairs):.3f}")
print(f"... between the bag of {NK} and one single fit: {np.mean([dm(m_bag, ms) for ms in m_seeds]):.3f}")
print(f"... between a fit with and without the four drifted sensors (the v10 -> v12 change): {dm(m_drift_all, m_seeds[0]):.3f}")

mean abs difference of test log-cost predictions between two single fits (different seeds): 0.133
... between the bag of 8 and one single fit: 0.088
... between a fit with and without the four drifted sensors (the v10 -> v12 change): 0.111


### 2. Classifier bag: out-of-fold probabilities for Platt calibration and test probabilities, averaged over 5 seeds

In [6]:
NC = 5
oof, pte = np.zeros(len(dev)), np.zeros(len(te))
for s in range(NC):
    prm = {**PARAMS, "random_state": SEED + s}
    o = fit_oof(dev[S], prm); p = xgb.XGBClassifier(**prm).fit(tr[S], tr.failure).predict_proba(te[S])[:, 1]
    if s == 0: o0, p0 = o, p                                           # seed 0 == the single-fit v2 classifier used before
    oof += o / NC; pte += p / NC
q, q1 = Platt().fit(oof, y).predict(pte), Platt().fit(o0, y).predict(p0)
print(f"mean calibrated test p: bag {q.mean():.4f} vs single fit {q1.mean():.4f}; correlation {np.corrcoef(q, q1)[0, 1]:.4f}")
print(f"OOF AUC bag {roc_auc_score(y, oof):.4f} vs single {roc_auc_score(y, o0):.4f}")

mean calibrated test p: bag 0.0203 vs single fit 0.0203; correlation 0.9958
OOF AUC bag 0.9841 vs single 0.9840


### 3. Decisions: same posterior-averaged shift, newsvendor budgets and flag rule as `v12_robust`

In [7]:
fl, bu = decide_pv(pavg(q), m_bag, resid_bag)
v12 = pd.read_csv("../submissions/submission_v12_robust.csv"); both = (v12.flag.to_numpy() == 1) & (fl == 1)
print(f"flag rate {fl.mean():.3%} (v12_robust {v12.flag.mean():.3%}); flags differing {int((v12.flag.to_numpy() != fl).sum())}; "
      f"budget change on commonly flagged: mean abs {np.abs(bu[both] - v12.budget.to_numpy()[both]).mean():.1f}")
write_submission(te.id, fl, bu, "v13_bag")
ref = pd.read_csv("../submissions/submission_v13_bag.csv")
assert (ref.flag.to_numpy() == fl).all() and np.allclose(ref.budget, bu.round(2), atol=0.011), "v13_bag not reproduced"

flag rate 8.795% (v12_robust 8.805%); flags differing 86; budget change on commonly flagged: mean abs 59.1
wrote outputs/submission_v13_bag.csv: flagged 8.79% of test vehicles


## Results and decisions

- **Seed noise is as large as the effect we measured.** Two single cost-model fits differ by ≈ 0.133 in mean absolute log-cost on the test depot (a bag of 8 is 0.088 from a single fit), more than the ≈ 0.111 that dropping the four drifted sensors changed (see v12). A single public-score difference between a one-seed v10_bayes and a one-seed v12_robust therefore cannot be attributed to the sensors.
- **`v13_bag`** changes only the averaging: 86 flags differ from v12_robust, mean absolute budget change ≈ 59 on commonly flagged vehicles. The classifier bag changes the ranking little (correlation 0.996 with the single fit).
- **Public score 17,974.55 vs 16,839.86 for v12_robust (+1,135).** With the same model family and inputs, averaging made the public score worse. The honest reading: v12_robust's lead over v10_bayes was largely a favourable draw of one seed, not a robust gain, and public differences of a few hundred to ≈ 1,000 points between sibling files are within noise at ~6,000 public vehicles.
- **Decision:** keep seed-averaging as the default for any final candidate (lower variance is what we want on the private 70%), but do **not** read the v12 → v13 score gap as evidence that averaging hurts either; it is one more draw.

**Caveats**
- Averaging reduces model variance only; it cannot remove the bias from the depot shift or the unknown test cost level.
- The shift posterior is reused from v10 (nine scores).